In [1]:
import gbd_mapping
import risk_distributions
import pathlib
import pandas as pd, numpy as np
import vivarium_inputs
from vivarium_inputs import utility_data, globals as vi_globals, utilities as vi_utils
from vivarium_gbd_access import gbd
import os, contextlib, warnings, loguru

from vivarium_inputs.validation.raw import DataDoesNotExistError, DataAbnormalError
from tqdm.notebook import tqdm

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
pd.set_option("display.max_columns", 30)

In [3]:
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

In [4]:
location = "india"
vehicle = "rice"
intervention_scenario = "intervention"

In [5]:
# Parameters
location = "nigeria"
vehicle = "bouillon"
intervention_scenario = "intervention"


In [6]:
index_cols = ["sex", "age_start", "age_end", "wealth_quintile"]

age_group_ids = [
    2,3,
    388,389,
    6,7,8,9,10,11,12,13,14,15,16,17,18,19,20, 30, 31, 32, 235
]
sex_ids = [1,2]

DRAWS = [f'draw_{i}' for i in range(500)] # NOTE: Some GBD 2021 things return 1,000 but others don't

In [7]:
fortification_hemoglobin_mean_difference = (
    pd.read_csv('../0100_data_prep/results/iron/fortification_hemoglobin_effects.csv')
        .set_index('vehicle_name').value
        .loc[vehicle]
)
fortification_hemoglobin_mean_difference

4.2

In [8]:
effective_baseline_coverage = (
    pd.read_csv(f'../0100_data_prep/results/iron/{vehicle}/baseline_fortification/effective_coverage/{location}.csv')
)
assert (effective_baseline_coverage.vehicle_name == vehicle).all()
effective_baseline_coverage = effective_baseline_coverage.drop(columns=["vehicle_name"])
effective_baseline_coverage

,wealth_quintile,value
0,fourth,0.535497
1,highest,0.559838
2,lowest,0.415369
3,middle,0.510484
4,second,0.467278


In [9]:
def expand(df):
    for col in sorted(list(set(df.columns) - {'value'})):
        if df[col].isnull().any():
            df = pd.concat([
                df[df[col].notnull()],
                *[df[df[col].isnull()].assign(**{col: value}) for value in df[df[col].notnull()][col].unique()]
            ])
    
    return df

In [10]:
for col, fill_value in [("age_start", 0), ("age_end", 125)]:
    if col not in effective_baseline_coverage.columns:
        effective_baseline_coverage[col] = fill_value
    else:
        effective_baseline_coverage[col] = effective_baseline_coverage[col].fillna(fill_value)

In [11]:
effective_baseline_coverage = expand(effective_baseline_coverage)
effective_baseline_coverage

,wealth_quintile,value,age_start,age_end
0,fourth,0.535497,0,125
1,highest,0.559838,0,125
2,lowest,0.415369,0,125
3,middle,0.510484,0,125
4,second,0.467278,0,125


In [12]:
effective_counterfactual_coverage = (
    pd.read_csv(f'../0100_data_prep/results/iron/{vehicle}/{intervention_scenario}/intervention_fortification/effective_coverage/{location}.csv')
)
assert (effective_counterfactual_coverage.vehicle_name == vehicle).all()
effective_counterfactual_coverage = effective_counterfactual_coverage.drop(columns=["vehicle_name"])
effective_counterfactual_coverage

,wealth_quintile,sex,value
0,lowest,Female,0.618069
1,second,Female,0.626300
2,middle,Female,0.627742
3,fourth,Female,0.632860
4,highest,Female,0.629615
5,lowest,Male,0.618069
6,second,Male,0.626300
7,middle,Male,0.627742
8,fourth,Male,0.632860
9,highest,Male,0.629615


In [13]:
for col, fill_value in [("age_start", 0), ("age_end", 125)]:
    if col not in effective_counterfactual_coverage.columns:
        effective_counterfactual_coverage[col] = fill_value
    else:
        effective_counterfactual_coverage[col] = effective_counterfactual_coverage[col].fillna(fill_value)

In [14]:
effective_counterfactual_coverage = expand(effective_counterfactual_coverage)
effective_counterfactual_coverage

,wealth_quintile,sex,value,age_start,age_end
0,lowest,Female,0.618069,0,125
1,second,Female,0.626300,0,125
2,middle,Female,0.627742,0,125
3,fourth,Female,0.632860,0,125
4,highest,Female,0.629615,0,125
5,lowest,Male,0.618069,0,125
6,second,Male,0.626300,0,125
7,middle,Male,0.627742,0,125
8,fourth,Male,0.632860,0,125
9,highest,Male,0.629615,0,125


In [15]:
non_pregnant_pop = (
    pd.read_csv(f'../0100_data_prep/results/population/stratified/{location}.csv')
).pipe(lambda df: df[df.pregnant == "not_pregnant"]).drop(columns="pregnant")
non_pregnant_pop = non_pregnant_pop.set_index([c for c in non_pregnant_pop.columns if c != 'value']).value
non_pregnant_pop

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    lowest             16960.496219
                               second             16890.884668
                               middle             15946.960379
                               fourth             14017.956071
                               highest            13029.810174
                                                      ...     
Male    95.0       125.000000  lowest              1846.968043
                               second              1616.329970
                               middle              1661.447043
                               fourth              1779.062429
                               highest             1896.973078
Name: value, Length: 250, dtype: float64

In [16]:
population_age_groups = non_pregnant_pop.reset_index()[["age_start", "age_end"]].drop_duplicates().sort_values("age_start")
population_age_groups

,age_start,age_end
0,0.000000,0.019178
5,0.019178,0.076712
10,0.076712,0.500000
15,0.500000,1.000000
20,1.000000,2.000000
25,2.000000,5.000000
30,5.000000,10.000000
35,10.000000,15.000000
40,15.000000,20.000000
45,20.000000,25.000000


In [17]:
def map_to_population_age_groups(df):
    return (
        population_age_groups.merge(df, how="cross", suffixes=("", "_orig"))
            .pipe(lambda df: df[(df.age_end <= df.age_end_orig) & (df.age_start >= df.age_start_orig)])
            .drop(columns=["age_start_orig", "age_end_orig"])
    )

In [18]:
effective_baseline_coverage = map_to_population_age_groups(effective_baseline_coverage).set_index([c for c in effective_baseline_coverage.columns if c != 'value']).value
effective_counterfactual_coverage = map_to_population_age_groups(effective_counterfactual_coverage).set_index([c for c in effective_counterfactual_coverage.columns if c != 'value']).value

In [19]:
delta_effective_coverage = effective_counterfactual_coverage.sub(effective_baseline_coverage)
delta_effective_coverage

wealth_quintile  age_start  age_end     sex   
fourth           0.000000   0.019178    Female    0.097363
                                        Male      0.097363
                 0.019178   0.076712    Female    0.097363
                                        Male      0.097363
                 0.076712   0.500000    Female    0.097363
                                                    ...   
second           85.000000  90.000000   Male      0.159021
                 90.000000  95.000000   Female    0.159021
                                        Male      0.159021
                 95.000000  125.000000  Female    0.159021
                                        Male      0.159021
Name: value, Length: 250, dtype: float64

In [20]:
def reshape_to_vivarium_format(df, location):
    df = vi_utils.reshape(df, value_cols=[c for c in df.columns if 'draw_' in c])
    df = vi_utils.scrub_gbd_conventions(df, location)
    df = vi_utils.split_interval(df, interval_column="age", split_column_prefix="age")
    df = vi_utils.split_interval(df, interval_column="year", split_column_prefix="year")
    df = vi_utils.sort_hierarchical_data(df)
    df.index = df.index.droplevel("location")
    return df

In [21]:
me_ids = {
    "hemoglobin_mean": 10487,
    "hemoglobin_sd": 10488,
}

In [22]:
location_id = utility_data.get_location_id(location.title())
hgb_mean = gbd.get_modelable_entity_draws(me_id=me_ids["hemoglobin_mean"], location_id=location_id, year_id=2021)
hgb_mean = reshape_to_vivarium_format(hgb_mean, location.title()).droplevel(["year_start", "year_end", "measure_id", "metric_id", "model_version_id", "modelable_entity_id"])[DRAWS].copy()
hgb_mean

draw_0      draw_1      draw_2      draw_3  \
sex    age_start age_end                                                      
Female 0.000000  0.019178    140.203331  139.700680  137.486100  143.738463   
       0.019178  0.076712    121.992596  122.837138  123.877519  120.604868   
       0.076712  0.500000    104.152465  104.343166  100.933682  103.672346   
       0.500000  1.000000    100.148802   99.844921   99.645127  100.145850   
       1.000000  2.000000    100.955723  100.339208  102.148597  101.234802   
       2.000000  5.000000    105.233662  104.333100  102.835176  105.304765   
       5.000000  10.000000   119.455834  112.957619  115.682929  114.256919   
       10.000000 15.000000   119.312919  106.273992  111.855854  122.097443   
       15.000000 20.000000   114.861226  117.794304  116.359602  117.648438   
       20.000000 25.000000   116.232257  117.251310  116.393998  114.894111   
       25.000000 30.000000   116.082332  114.626063  116.408492  116.811736   
       30.000000 35.000000   115.103530  115.163515  116.767807  116.269128   
       35.000000 40.000000   115.986054  113.528076  114.852973  114.288218   
       40.000000 45.000000   115.257500  117.940598  117.278455  116.853328   
       45.000000 50.000000   118.568757  119.048450  117.278407  117.835148   
       50.000000 55.000000   115.669641  121.096756  111.909301  123.510802   
       55.000000 60.000000   116.888277  111.934668  117.516013  116.335911   
       60.000000 65.000000   117.073529  120.448205  112.138824  119.361783   
       65.000000 70.000000   120.936133  131.305616  125.299052  121.720141   
       70.000000 75.000000   120.728706  120.069195  124.432953  117.908148   
       75.000000 80.000000   126.020295  126.733554  121.556975  127.985198   
       80.000000 85.000000   126.158836  126.248607  116.366249  123.473284   
       85.000000 90.000000   118.780034  118.418548  114.061451  115.310723   
       90.000000 95.000000   109.477010  104.789317  110.610979  108.944244   
       95.000000 125.000000  102.971577  100.970692   99.657313   98.358658   
Male   0.000000  0.019178    139.655000  152.652888  141.405358  135.236759   
       0.019178  0.076712    111.875330  122.227176  119.526917  118.369764   
       0.076712  0.500000     94.267052   98.335982   96.967303   98.720429   
       0.500000  1.000000    100.092403   98.678200   93.913129   99.908232   
       1.000000  2.000000    100.226883   94.470196   98.331789   95.255325   
       2.000000  5.000000    101.069093  107.942797  103.555610  102.374253   
       5.000000  10.000000   119.838610  110.970261  109.073412  112.143982   
       10.000000 15.000000   129.595152  119.217013  127.657332  129.642930   
       15.000000 20.000000   129.129905  144.220990  141.441577  142.995724   
       20.000000 25.000000   151.050744  131.461208  146.612140  132.307327   
       25.000000 30.000000   132.425569  149.591271  142.560174  135.773968   
       30.000000 35.000000   140.605288  141.846950  135.968545  160.662459   
       35.000000 40.000000   138.202586  145.354328  144.617087  133.079794   
       40.000000 45.000000   141.806468  145.535717  136.109962  148.130476   
       45.000000 50.000000   133.416092  136.101393  150.337119  141.486861   
       50.000000 55.000000   144.695059  135.210554  139.837715  139.728713   
       55.000000 60.000000   138.257682  142.365263  142.266348  143.671582   
       60.000000 65.000000   137.633530  133.990371  143.313102  140.484227   
       65.000000 70.000000   143.947810  146.078428  140.887869  134.047618   
       70.000000 75.000000   141.462864  144.115084  130.980064  145.225366   
       75.000000 80.000000   125.293565  127.635808  121.016674  127.115747   
       80.000000 85.000000   136.438006  140.853010  131.367813  139.943361   
       85.000000 90.000000   133.029351  120.328832  136.611180  126.391697   
       90.000000 95.000000   116.866389  104.222078  111.737153  113.207507   
    

In [23]:
hemoglobin_mean_disparities = pd.read_csv(f'../0100_data_prep/results/hemoglobin/mean_disparities/{location}.csv')
hemoglobin_mean_disparities = (
    map_to_population_age_groups(hemoglobin_mean_disparities[hemoglobin_mean_disparities.pregnant == "not_pregnant"].drop(columns=["pregnant"]))
        .set_index(["sex", "age_start", "age_end", "wealth_quintile"]).value
)
hemoglobin_mean_disparities

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    lowest              97.093264
                               second              99.647180
                               middle             102.658490
                               fourth             103.792784
                               highest            107.778540
                                                     ...    
Male    95.0       125.000000  lowest             113.444966
                               second             115.420130
                               middle             116.087185
                               fourth             116.578539
                               highest            118.793246
Name: value, Length: 250, dtype: float64

In [24]:
wealth_quintile_probabilities = pd.read_csv(f'../0100_data_prep/results/wealth_quintile_probabilities/{location}.csv')
wealth_quintile_probabilities

,sex,age_start,age_end,pregnant,lowest,second,middle,fourth,highest
0,Female,0.0,5.0,not_pregnant,0.220707,0.219801,0.207518,0.182416,0.169557
1,Female,5.0,15.0,not_pregnant,0.222903,0.211314,0.200941,0.189893,0.174949
2,Female,15.0,30.0,not_pregnant,0.165248,0.194708,0.200403,0.223650,0.215992
3,Female,15.0,30.0,pregnant,0.229568,0.269087,0.218290,0.169175,0.113880
4,Female,30.0,50.0,not_pregnant,0.167414,0.176150,0.188366,0.214621,0.253449
5,Female,30.0,50.0,pregnant,0.247900,0.197054,0.182014,0.163763,0.209268
6,Female,50.0,125.0,not_pregnant,0.196077,0.184902,0.228885,0.192974,0.197161
7,Male,0.0,5.0,not_pregnant,0.214963,0.223211,0.203137,0.188194,0.170495
8,Male,5.0,15.0,not_pregnant,0.227134,0.211917,0.202795,0.189340,0.168813
9,Male,15.0,30.0,not_pregnant,0.194480,0.197231,0.196587,0.211924,0.199779


In [25]:
wealth_quintile_probabilities = (
    map_to_population_age_groups(wealth_quintile_probabilities[wealth_quintile_probabilities.pregnant == "not_pregnant"].drop(columns=["pregnant"]))
        .set_index(["sex", "age_start", "age_end"])
)
wealth_quintile_probabilities.columns.name = 'wealth_quintile'
wealth_quintile_probabilities = wealth_quintile_probabilities.stack()
wealth_quintile_probabilities

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    lowest             0.220707
                               second             0.219801
                               middle             0.207518
                               fourth             0.182416
                               highest            0.169557
                                                    ...   
Male    95.0       125.000000  lowest             0.209864
                               second             0.183658
                               middle             0.188784
                               fourth             0.202148
                               highest            0.215546
Length: 250, dtype: float64

In [26]:
assert np.allclose(wealth_quintile_probabilities.groupby(["sex", "age_start", "age_end"]).sum(), 1.0)

In [27]:
pre_disparity_groups = hgb_mean.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in hgb_mean.index.names if c != 'wealth_quintile']).sum()
pre_disparity_groups

draw_0      draw_1      draw_2      draw_3  \
sex    age_start age_end                                                      
Female 0.000000  0.019178    140.203331  139.700680  137.486100  143.738463   
       0.019178  0.076712    121.992596  122.837138  123.877519  120.604868   
       0.076712  0.500000    104.152465  104.343166  100.933682  103.672346   
       0.500000  1.000000    100.148802   99.844921   99.645127  100.145850   
       1.000000  2.000000    100.955723  100.339208  102.148597  101.234802   
       2.000000  5.000000    105.233662  104.333100  102.835176  105.304765   
       5.000000  10.000000   119.455834  112.957619  115.682929  114.256919   
       10.000000 15.000000   119.312919  106.273992  111.855854  122.097443   
       15.000000 20.000000   114.861226  117.794304  116.359602  117.648438   
       20.000000 25.000000   116.232257  117.251310  116.393998  114.894111   
       25.000000 30.000000   116.082332  114.626063  116.408492  116.811736   
       30.000000 35.000000   115.103530  115.163515  116.767807  116.269128   
       35.000000 40.000000   115.986054  113.528076  114.852973  114.288218   
       40.000000 45.000000   115.257500  117.940598  117.278455  116.853328   
       45.000000 50.000000   118.568757  119.048450  117.278407  117.835148   
       50.000000 55.000000   115.669641  121.096756  111.909301  123.510802   
       55.000000 60.000000   116.888277  111.934668  117.516013  116.335911   
       60.000000 65.000000   117.073529  120.448205  112.138824  119.361783   
       65.000000 70.000000   120.936133  131.305616  125.299052  121.720141   
       70.000000 75.000000   120.728706  120.069195  124.432953  117.908148   
       75.000000 80.000000   126.020295  126.733554  121.556975  127.985198   
       80.000000 85.000000   126.158836  126.248607  116.366249  123.473284   
       85.000000 90.000000   118.780034  118.418548  114.061451  115.310723   
       90.000000 95.000000   109.477010  104.789317  110.610979  108.944244   
       95.000000 125.000000  102.971577  100.970692   99.657313   98.358658   
Male   0.000000  0.019178    139.655000  152.652888  141.405358  135.236759   
       0.019178  0.076712    111.875330  122.227176  119.526917  118.369764   
       0.076712  0.500000     94.267052   98.335982   96.967303   98.720429   
       0.500000  1.000000    100.092403   98.678200   93.913129   99.908232   
       1.000000  2.000000    100.226883   94.470196   98.331789   95.255325   
       2.000000  5.000000    101.069093  107.942797  103.555610  102.374253   
       5.000000  10.000000   119.838610  110.970261  109.073412  112.143982   
       10.000000 15.000000   129.595152  119.217013  127.657332  129.642930   
       15.000000 20.000000   129.129905  144.220990  141.441577  142.995724   
       20.000000 25.000000   151.050744  131.461208  146.612140  132.307327   
       25.000000 30.000000   132.425569  149.591271  142.560174  135.773968   
       30.000000 35.000000   140.605288  141.846950  135.968545  160.662459   
       35.000000 40.000000   138.202586  145.354328  144.617087  133.079794   
       40.000000 45.000000   141.806468  145.535717  136.109962  148.130476   
       45.000000 50.000000   133.416092  136.101393  150.337119  141.486861   
       50.000000 55.000000   144.695059  135.210554  139.837715  139.728713   
       55.000000 60.000000   138.257682  142.365263  142.266348  143.671582   
       60.000000 65.000000   137.633530  133.990371  143.313102  140.484227   
       65.000000 70.000000   143.947810  146.078428  140.887869  134.047618   
       70.000000 75.000000   141.462864  144.115084  130.980064  145.225366   
       75.000000 80.000000   125.293565  127.635808  121.016674  127.115747   
       80.000000 85.000000   136.438006  140.853010  131.367813  139.943361   
       85.000000 90.000000   133.029351  120.328832  136.611180  126.391697   
       90.000000 95.000000   116.866389  104.222078  111.737153  113.207507   
    

In [28]:
hgb_mean = hgb_mean.mul(hemoglobin_mean_disparities, axis=0)

In [29]:
scale_factor = pre_disparity_groups / hgb_mean.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in hgb_mean.index.names if c != 'wealth_quintile']).sum()
scale_factor

draw_0    draw_1    draw_2    draw_3    draw_4  \
sex    age_start age_end                                                        
Female 0.000000  0.019178    0.009819  0.009819  0.009819  0.009819  0.009819   
       0.019178  0.076712    0.009819  0.009819  0.009819  0.009819  0.009819   
       0.076712  0.500000    0.009819  0.009819  0.009819  0.009819  0.009819   
       0.500000  1.000000    0.009819  0.009819  0.009819  0.009819  0.009819   
       1.000000  2.000000    0.009819  0.009819  0.009819  0.009819  0.009819   
       2.000000  5.000000    0.009819  0.009819  0.009819  0.009819  0.009819   
       5.000000  10.000000   0.009182  0.009182  0.009182  0.009182  0.009182   
       10.000000 15.000000   0.009182  0.009182  0.009182  0.009182  0.009182   
       15.000000 20.000000   0.008605  0.008605  0.008605  0.008605  0.008605   
       20.000000 25.000000   0.008605  0.008605  0.008605  0.008605  0.008605   
       25.000000 30.000000   0.008605  0.008605  0.008605  0.008605  0.008605   
       30.000000 35.000000   0.008597  0.008597  0.008597  0.008597  0.008597   
       35.000000 40.000000   0.008597  0.008597  0.008597  0.008597  0.008597   
       40.000000 45.000000   0.008597  0.008597  0.008597  0.008597  0.008597   
       45.000000 50.000000   0.008597  0.008597  0.008597  0.008597  0.008597   
       50.000000 55.000000   0.008615  0.008615  0.008615  0.008615  0.008615   
       55.000000 60.000000   0.008615  0.008615  0.008615  0.008615  0.008615   
       60.000000 65.000000   0.008615  0.008615  0.008615  0.008615  0.008615   
       65.000000 70.000000   0.008615  0.008615  0.008615  0.008615  0.008615   
       70.000000 75.000000   0.008615  0.008615  0.008615  0.008615  0.008615   
       75.000000 80.000000   0.008615  0.008615  0.008615  0.008615  0.008615   
       80.000000 85.000000   0.008615  0.008615  0.008615  0.008615  0.008615   
       85.000000 90.000000   0.008615  0.008615  0.008615  0.008615  0.008615   
       90.000000 95.000000   0.008615  0.008615  0.008615  0.008615  0.008615   
       95.000000 125.000000  0.008615  0.008615  0.008615  0.008615  0.008615   
Male   0.000000  0.019178    0.009962  0.009962  0.009962  0.009962  0.009962   
       0.019178  0.076712    0.009962  0.009962  0.009962  0.009962  0.009962   
       0.076712  0.500000    0.009962  0.009962  0.009962  0.009962  0.009962   
       0.500000  1.000000    0.009962  0.009962  0.009962  0.009962  0.009962   
       1.000000  2.000000    0.009962  0.009962  0.009962  0.009962  0.009962   
       2.000000  5.000000    0.009962  0.009962  0.009962  0.009962  0.009962   
       5.000000  10.000000   0.009250  0.009250  0.009250  0.009250  0.009250   
       10.000000 15.000000   0.009250  0.009250  0.009250  0.009250  0.009250   
       15.000000 20.000000   0.008614  0.008614  0.008614  0.008614  0.008614   
       20.000000 25.000000   0.008614  0.008614  0.008614  0.008614  0.008614   
       25.000000 30.000000   0.008614  0.008614  0.008614  0.008614  0.008614   
       30.000000 35.000000   0.008591  0.008591  0.008591  0.008591  0.008591   
       35.000000 40.000000   0.008591  0.008591  0.008591  0.008591  0.008591   
       40.000000 45.000000   0.008591  0.008591  0.008591  0.008591  0.008591   
       45.000000 50.000000   0.008591  0.008591  0.008591  0.008591  0.008591   
       50.000000 55.000000   0.008614  0.008614  0.008614  0.008614  0.008614   
       55.000000 60.000000   0.008614  0.008614  0.008614  0.008614  0.008614   
       60.000000 65.000000   0.008614  0.008614  0.008614  0.008614  0.008614   
       65.000000 70.000000   0.008614  0.008614  0.008614  0.008614  0.008614   
       70.000000 75.000000   0.008614  0.008614  0.008614  0.008614  0.008614   
       75.000000 80.000000   0.008614  0.008614  0.008614  0.008614  0.008614   
       80.000000 85.000000   0.008614  0.008614  0.008614  0.008614  0.008614   
       85.000000 90.000000   0.008614  0.008614  0.008614  0.0

In [30]:
hgb_mean = hgb_mean * scale_factor

In [31]:
assert np.allclose(
    hgb_mean.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in hgb_mean.index.names if c != 'wealth_quintile']).sum(),
    pre_disparity_groups,
)

In [32]:
counterfactual_hgb_mean = hgb_mean.add(delta_effective_coverage * fortification_hemoglobin_mean_difference, axis=0)
counterfactual_hgb_mean

draw_0      draw_1  \
sex    age_start age_end    wealth_quintile                           
Female 0.0       0.019178   fourth           143.295931  142.783659   
                            highest          148.667085  148.135141   
                            lowest           134.515409  134.036202   
                            middle           141.817956  141.311282   
                            second           137.847824  137.356013   
...                                                 ...         ...   
Male   95.0      125.000000 fourth           105.260577  104.955276   
                            highest          107.136639  106.825538   
                            lowest           102.884631  102.587537   
                            middle           104.902207  104.598194   
                            second           104.477659  104.175392   

                                                 draw_2      draw_3  \
sex    age_start age_end    wealth_quintile                           
Female 0.0       0.019178   fourth           140.526689  146.898731   
                            highest          145.791501  152.408236   
                            lowest           131.924913  137.885659   
                            middle           139.078978  145.381383   
                            second           135.189189  141.306724   
...                                                 ...         ...   
Male   95.0      125.000000 fourth           100.307472  103.727010   
                            highest          102.089437  105.573938   
                            lowest            98.064663  101.392286   
                            middle            99.969979  103.375104   
                            second            99.573772  102.959331   

                                                 draw_4      draw_5  \
sex    age_start age_end    wealth_quintile                           
Female 0.0       0.019178   fourth           151.875902  146.042591   
                            highest          157.576537  151.519220   
                            lowest           142.541568  137.084780   
                            middle           150.304162  144.534600   
                            second           146.085101  140.484779   
...                                                 ...         ...   
Male   95.0      125.000000 fourth           108.462520  106.309486   
                            highest          110.399412  108.205475   
                            lowest           106.000508  103.905347   
                            middle           108.090656  105.946696   
                            second           107.647786  105.516146   

                                                 draw_6      draw_7  \
sex    age_start age_end    wealth_quintile                           
Female 0.0       0.019178   fourth           138.889582  150.080837   
                            highest          144.091528  155.712539   
                            lowest           130.393477  140.862369   
                            middle           137.459762  148.528714   
                            second           133.617470  144.361733   
...                                                 ...         ...   
Male   95.0      125.000000 fourth           104.096555  111.326653   
                            highest          105.950503  113.317955   
                            lowest           101.751898  108.787654   
                            middle           103.743092  110.942716   
                            second           103.325204  110.483458   

                                                 draw_8      draw_9  \
sex    age_start age_end    wealth_quintile                           
Female 0.0       0.019178   fourth           151.691866  139.622355   
                            highest          157.385434  144.852440   
                            lowest           142.369412  131.078952  

In [33]:
hgb_mean.columns.name = "draw"
hgb_mean = hgb_mean.stack().rename("mean").reset_index()

In [34]:
counterfactual_hgb_mean.columns.name = "draw"
counterfactual_hgb_mean = counterfactual_hgb_mean.stack().rename("mean").reset_index()

In [35]:
location_id = utility_data.get_location_id(location.title())
hgb_sd = gbd.get_modelable_entity_draws(me_id=me_ids["hemoglobin_sd"], location_id=location_id, year_id=2021)
hgb_sd = reshape_to_vivarium_format(hgb_sd, location.title()).droplevel(["year_start", "year_end", "measure_id", "metric_id", "model_version_id", "modelable_entity_id"])[DRAWS].copy()
hgb_sd

draw_0     draw_1     draw_2     draw_3  \
sex    age_start age_end                                                  
Female 0.000000  0.019178    26.026960  23.907182  16.403169  18.715768   
       0.019178  0.076712    29.653984  25.132357  26.064191  35.471576   
       0.076712  0.500000    22.493218  16.259081  17.452182  15.476258   
       0.500000  1.000000    15.088749  16.763458  14.467460  14.968448   
       1.000000  2.000000    15.397715  16.467520  16.931422  15.316047   
       2.000000  5.000000    15.553860  13.236368  15.314247  15.205571   
       5.000000  10.000000   24.246181   8.464774  13.511319  12.174676   
       10.000000 15.000000   20.372010  17.600339  15.670304  18.874888   
       15.000000 20.000000   13.849848  15.303821  13.412809  15.645587   
       20.000000 25.000000   14.161841  13.816199  16.113706  13.064190   
       25.000000 30.000000   13.951453  12.087523  13.960027  15.747683   
       30.000000 35.000000   15.124416  13.912934  14.508748  14.284332   
       35.000000 40.000000   14.537795  12.796153  11.977138  12.619102   
       40.000000 45.000000   15.511511  15.320203  13.814969  15.477653   
       45.000000 50.000000   14.740887  16.461188  14.795978  14.266428   
       50.000000 55.000000   12.005022  19.946933  14.359695  22.342577   
       55.000000 60.000000   15.457708   9.403534  17.510417  14.185617   
       60.000000 65.000000   17.870134  23.535264  15.190874  19.820442   
       65.000000 70.000000   18.274735  38.392728  27.115780  22.460828   
       70.000000 75.000000   14.297770  12.642415  17.848045  11.580773   
       75.000000 80.000000   15.552421  25.187471   9.408474  17.651964   
       80.000000 85.000000   27.470516  25.512507  18.952878  25.817333   
       85.000000 90.000000   18.088001  20.580833   5.178689  16.104535   
       90.000000 95.000000   12.785354  20.398206  15.979831  13.495828   
       95.000000 125.000000  23.461384  26.340875  35.452001  34.646552   
Male   0.000000  0.019178     8.741365   8.444370  10.058485   9.269974   
       0.019178  0.076712    45.945438  28.232303  33.884957  25.657119   
       0.076712  0.500000    10.834561  14.464839  16.716692  13.781233   
       0.500000  1.000000    17.866906  15.667957  15.926898  17.670570   
       1.000000  2.000000    17.452899  16.689185  13.466597  16.075586   
       2.000000  5.000000    11.124414  20.798399  13.040816  21.819873   
       5.000000  10.000000   15.571013  12.788589  13.163620  10.470561   
       10.000000 15.000000   20.593272   7.675260  11.134538  13.284772   
       15.000000 20.000000   11.081504  22.556104  22.074298  21.838462   
       20.000000 25.000000   18.281512   7.567572  14.716066   9.559315   
       25.000000 30.000000    9.776189  17.322972  12.849834   7.167079   
       30.000000 35.000000   12.832856  14.604296   8.867764  27.680958   
       35.000000 40.000000    7.747828  16.962800  16.855032  10.901524   
       40.000000 45.000000   11.482526  14.684343   8.398772  22.837871   
       45.000000 50.000000   12.619049  13.035039  20.963212  15.980802   
       50.000000 55.000000   15.864704  16.779504  15.424543  21.599276   
       55.000000 60.000000   10.580076  27.129333  19.432159  17.335354   
       60.000000 65.000000   15.722404  12.879010  20.046828  24.112250   
       65.000000 70.000000   21.271206  16.053375  16.248970  17.207534   
       70.000000 75.000000   24.062525  24.232873  13.549617  18.340322   
       75.000000 80.000000    9.509731  13.080280   8.034360   9.796766   
       80.000000 85.000000   20.360018  15.849824  13.075308  21.846383   
       85.000000 90.000000   30.679256  12.187568  24.214234  24.380637   
       90.000000 95.000000   12.432987  22.692279   9.217534   6.586342   
       95.000000 125.000000  25.841204  20.160131  32.834640  21.792155   

                                draw_4     draw_5     draw_6     draw_7  \
sex    age_start age_end                                

In [36]:
hemoglobin_sd_disparities = pd.read_csv(f'../0100_data_prep/results/hemoglobin/sd_disparities/{location}.csv')
hemoglobin_sd_disparities = (
    map_to_population_age_groups(hemoglobin_sd_disparities[hemoglobin_sd_disparities.pregnant == "not_pregnant"].drop(columns=["pregnant"]))
        .set_index(["sex", "age_start", "age_end", "wealth_quintile"]).value
)
hemoglobin_sd_disparities

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    lowest             16.607283
                               second             15.898899
                               middle             15.268520
                               fourth             14.286274
                               highest            13.194608
                                                    ...    
Male    95.0       125.000000  lowest             15.148809
                               second             14.920351
                               middle             15.108541
                               fourth             14.792239
                               highest            14.222341
Name: value, Length: 250, dtype: float64

In [37]:
pre_disparity_groups = hgb_sd.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in hgb_sd.index.names if c != 'wealth_quintile']).sum()
pre_disparity_groups

draw_0     draw_1     draw_2     draw_3  \
sex    age_start age_end                                                  
Female 0.000000  0.019178    26.026960  23.907182  16.403169  18.715768   
       0.019178  0.076712    29.653984  25.132357  26.064191  35.471576   
       0.076712  0.500000    22.493218  16.259081  17.452182  15.476258   
       0.500000  1.000000    15.088749  16.763458  14.467460  14.968448   
       1.000000  2.000000    15.397715  16.467520  16.931422  15.316047   
       2.000000  5.000000    15.553860  13.236368  15.314247  15.205571   
       5.000000  10.000000   24.246181   8.464774  13.511319  12.174676   
       10.000000 15.000000   20.372010  17.600339  15.670304  18.874888   
       15.000000 20.000000   13.849848  15.303821  13.412809  15.645587   
       20.000000 25.000000   14.161841  13.816199  16.113706  13.064190   
       25.000000 30.000000   13.951453  12.087523  13.960027  15.747683   
       30.000000 35.000000   15.124416  13.912934  14.508748  14.284332   
       35.000000 40.000000   14.537795  12.796153  11.977138  12.619102   
       40.000000 45.000000   15.511511  15.320203  13.814969  15.477653   
       45.000000 50.000000   14.740887  16.461188  14.795978  14.266428   
       50.000000 55.000000   12.005022  19.946933  14.359695  22.342577   
       55.000000 60.000000   15.457708   9.403534  17.510417  14.185617   
       60.000000 65.000000   17.870134  23.535264  15.190874  19.820442   
       65.000000 70.000000   18.274735  38.392728  27.115780  22.460828   
       70.000000 75.000000   14.297770  12.642415  17.848045  11.580773   
       75.000000 80.000000   15.552421  25.187471   9.408474  17.651964   
       80.000000 85.000000   27.470516  25.512507  18.952878  25.817333   
       85.000000 90.000000   18.088001  20.580833   5.178689  16.104535   
       90.000000 95.000000   12.785354  20.398206  15.979831  13.495828   
       95.000000 125.000000  23.461384  26.340875  35.452001  34.646552   
Male   0.000000  0.019178     8.741365   8.444370  10.058485   9.269974   
       0.019178  0.076712    45.945438  28.232303  33.884957  25.657119   
       0.076712  0.500000    10.834561  14.464839  16.716692  13.781233   
       0.500000  1.000000    17.866906  15.667957  15.926898  17.670570   
       1.000000  2.000000    17.452899  16.689185  13.466597  16.075586   
       2.000000  5.000000    11.124414  20.798399  13.040816  21.819873   
       5.000000  10.000000   15.571013  12.788589  13.163620  10.470561   
       10.000000 15.000000   20.593272   7.675260  11.134538  13.284772   
       15.000000 20.000000   11.081504  22.556104  22.074298  21.838462   
       20.000000 25.000000   18.281512   7.567572  14.716066   9.559315   
       25.000000 30.000000    9.776189  17.322972  12.849834   7.167079   
       30.000000 35.000000   12.832856  14.604296   8.867764  27.680958   
       35.000000 40.000000    7.747828  16.962800  16.855032  10.901524   
       40.000000 45.000000   11.482526  14.684343   8.398772  22.837871   
       45.000000 50.000000   12.619049  13.035039  20.963212  15.980802   
       50.000000 55.000000   15.864704  16.779504  15.424543  21.599276   
       55.000000 60.000000   10.580076  27.129333  19.432159  17.335354   
       60.000000 65.000000   15.722404  12.879010  20.046828  24.112250   
       65.000000 70.000000   21.271206  16.053375  16.248970  17.207534   
       70.000000 75.000000   24.062525  24.232873  13.549617  18.340322   
       75.000000 80.000000    9.509731  13.080280   8.034360   9.796766   
       80.000000 85.000000   20.360018  15.849824  13.075308  21.846383   
       85.000000 90.000000   30.679256  12.187568  24.214234  24.380637   
       90.000000 95.000000   12.432987  22.692279   9.217534   6.586342   
       95.000000 125.000000  25.841204  20.160131  32.834640  21.792155   

                                draw_4     draw_5     draw_6     draw_7  \
sex    age_start age_end                                

In [38]:
hgb_sd = hgb_sd.mul(hemoglobin_sd_disparities, axis=0)

In [39]:
scale_factor = pre_disparity_groups / hgb_sd.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in hgb_sd.index.names if c != 'wealth_quintile']).sum()
scale_factor

draw_0    draw_1    draw_2    draw_3    draw_4  \
sex    age_start age_end                                                        
Female 0.000000  0.019178    0.065912  0.065912  0.065912  0.065912  0.065912   
       0.019178  0.076712    0.065912  0.065912  0.065912  0.065912  0.065912   
       0.076712  0.500000    0.065912  0.065912  0.065912  0.065912  0.065912   
       0.500000  1.000000    0.065912  0.065912  0.065912  0.065912  0.065912   
       1.000000  2.000000    0.065912  0.065912  0.065912  0.065912  0.065912   
       2.000000  5.000000    0.065912  0.065912  0.065912  0.065912  0.065912   
       5.000000  10.000000   0.066637  0.066637  0.066637  0.066637  0.066637   
       10.000000 15.000000   0.066637  0.066637  0.066637  0.066637  0.066637   
       15.000000 20.000000   0.067493  0.067493  0.067493  0.067493  0.067493   
       20.000000 25.000000   0.067493  0.067493  0.067493  0.067493  0.067493   
       25.000000 30.000000   0.067493  0.067493  0.067493  0.067493  0.067493   
       30.000000 35.000000   0.067615  0.067615  0.067615  0.067615  0.067615   
       35.000000 40.000000   0.067615  0.067615  0.067615  0.067615  0.067615   
       40.000000 45.000000   0.067615  0.067615  0.067615  0.067615  0.067615   
       45.000000 50.000000   0.067615  0.067615  0.067615  0.067615  0.067615   
       50.000000 55.000000   0.067359  0.067359  0.067359  0.067359  0.067359   
       55.000000 60.000000   0.067359  0.067359  0.067359  0.067359  0.067359   
       60.000000 65.000000   0.067359  0.067359  0.067359  0.067359  0.067359   
       65.000000 70.000000   0.067359  0.067359  0.067359  0.067359  0.067359   
       70.000000 75.000000   0.067359  0.067359  0.067359  0.067359  0.067359   
       75.000000 80.000000   0.067359  0.067359  0.067359  0.067359  0.067359   
       80.000000 85.000000   0.067359  0.067359  0.067359  0.067359  0.067359   
       85.000000 90.000000   0.067359  0.067359  0.067359  0.067359  0.067359   
       90.000000 95.000000   0.067359  0.067359  0.067359  0.067359  0.067359   
       95.000000 125.000000  0.067359  0.067359  0.067359  0.067359  0.067359   
Male   0.000000  0.019178    0.066669  0.066669  0.066669  0.066669  0.066669   
       0.019178  0.076712    0.066669  0.066669  0.066669  0.066669  0.066669   
       0.076712  0.500000    0.066669  0.066669  0.066669  0.066669  0.066669   
       0.500000  1.000000    0.066669  0.066669  0.066669  0.066669  0.066669   
       1.000000  2.000000    0.066669  0.066669  0.066669  0.066669  0.066669   
       2.000000  5.000000    0.066669  0.066669  0.066669  0.066669  0.066669   
       5.000000  10.000000   0.066943  0.066943  0.066943  0.066943  0.066943   
       10.000000 15.000000   0.066943  0.066943  0.066943  0.066943  0.066943   
       15.000000 20.000000   0.067407  0.067407  0.067407  0.067407  0.067407   
       20.000000 25.000000   0.067407  0.067407  0.067407  0.067407  0.067407   
       25.000000 30.000000   0.067407  0.067407  0.067407  0.067407  0.067407   
       30.000000 35.000000   0.067687  0.067687  0.067687  0.067687  0.067687   
       35.000000 40.000000   0.067687  0.067687  0.067687  0.067687  0.067687   
       40.000000 45.000000   0.067687  0.067687  0.067687  0.067687  0.067687   
       45.000000 50.000000   0.067687  0.067687  0.067687  0.067687  0.067687   
       50.000000 55.000000   0.067442  0.067442  0.067442  0.067442  0.067442   
       55.000000 60.000000   0.067442  0.067442  0.067442  0.067442  0.067442   
       60.000000 65.000000   0.067442  0.067442  0.067442  0.067442  0.067442   
       65.000000 70.000000   0.067442  0.067442  0.067442  0.067442  0.067442   
       70.000000 75.000000   0.067442  0.067442  0.067442  0.067442  0.067442   
       75.000000 80.000000   0.067442  0.067442  0.067442  0.067442  0.067442   
       80.000000 85.000000   0.067442  0.067442  0.067442  0.067442  0.067442   
       85.000000 90.000000   0.067442  0.067442  0.067442  0.0

In [40]:
hgb_sd = hgb_sd * scale_factor

In [41]:
assert np.allclose(
    hgb_sd.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in hgb_sd.index.names if c != 'wealth_quintile']).sum(),
    pre_disparity_groups,
)

In [42]:
hgb_sd.columns.name = "draw"
hgb_sd = hgb_sd.stack().rename("sd").reset_index()
hgb_sd

,sex,age_start,age_end,wealth_quintile,draw,sd
0,Female,0.0,0.019178,lowest,draw_0,28.489640
1,Female,0.0,0.019178,lowest,draw_1,26.169288
2,Female,0.0,0.019178,lowest,draw_2,17.955243
3,Female,0.0,0.019178,lowest,draw_3,20.486661
4,Female,0.0,0.019178,lowest,draw_4,18.648197
...,...,...,...,...,...,...
124995,Male,95.0,125.000000,highest,draw_495,33.780207
124996,Male,95.0,125.000000,highest,draw_496,18.376878
124997,Male,95.0,125.000000,highest,draw_497,22.113952
124998,Male,95.0,125.000000,highest,draw_498,35.369247


In [43]:
mean_and_sd_hgb = pd.concat([
    hgb_mean.merge(hgb_sd, how="outer", validate="m:1").assign(scenario='baseline'),
    counterfactual_hgb_mean.merge(hgb_sd, how="outer", validate="m:1").assign(scenario='intervention')
])
mean_and_sd_hgb

,sex,age_start,age_end,wealth_quintile,draw,mean,sd,scenario
0,Female,0.0,0.019178,lowest,draw_0,133.664070,28.489640,baseline
1,Female,0.0,0.019178,lowest,draw_1,133.184863,26.169288,baseline
2,Female,0.0,0.019178,lowest,draw_2,131.073574,17.955243,baseline
3,Female,0.0,0.019178,lowest,draw_3,137.034319,20.486661,baseline
4,Female,0.0,0.019178,lowest,draw_4,141.690229,18.648197,baseline
...,...,...,...,...,...,...,...,...
124995,Male,95.0,125.000000,second,draw_495,97.031167,35.438084,intervention
124996,Male,95.0,125.000000,second,draw_496,106.139058,19.278785,intervention
124997,Male,95.0,125.000000,second,draw_497,105.312300,23.199269,intervention
124998,Male,95.0,125.000000,second,draw_498,99.292330,37.105111,intervention


In [44]:
thresholds = reshape_to_vivarium_format(pd.read_csv('/share/mnch/anemia/code/reference/model/anemia_thresholds.csv'), location.title()).droplevel(["age_group_name", "grp"]).reset_index()
thresholds

,sex,age_start,age_end,hgb_lower_anemic,hgb_lower_mild,hgb_lower_moderate,hgb_lower_severe,hgb_upper_anemic,hgb_upper_mild,hgb_upper_moderate,hgb_upper_severe,pregnant
0,Female,0.000000,0.019178,0,145,100,0,160,160,145,100,0
1,Female,0.019178,0.076712,0,120,85,0,135,135,120,85,0
2,Female,0.076712,0.500000,0,100,70,0,110,110,100,70,0
3,Female,0.500000,1.000000,0,100,70,0,110,110,100,70,0
4,Female,1.000000,2.000000,0,100,70,0,110,110,100,70,0
5,Female,2.000000,5.000000,0,100,70,0,110,110,100,70,0
6,Female,5.000000,10.000000,0,110,80,0,115,115,110,80,0
7,Female,10.000000,15.000000,0,100,70,0,110,110,100,70,1
8,Female,10.000000,15.000000,0,110,80,0,115,115,110,80,0
9,Female,15.000000,20.000000,0,110,80,0,120,120,110,80,0


In [45]:
mean_and_sd_hgb = mean_and_sd_hgb.assign(pregnant=0).merge(thresholds, on=["sex", "age_start", "age_end", "pregnant"], how="left", validate="m:1")
mean_and_sd_hgb

,sex,age_start,age_end,wealth_quintile,draw,mean,sd,scenario,pregnant,hgb_lower_anemic,hgb_lower_mild,hgb_lower_moderate,hgb_lower_severe,hgb_upper_anemic,hgb_upper_mild,hgb_upper_moderate,hgb_upper_severe
0,Female,0.0,0.019178,lowest,draw_0,133.664070,28.489640,baseline,0,0,145,100,0,160,160,145,100
1,Female,0.0,0.019178,lowest,draw_1,133.184863,26.169288,baseline,0,0,145,100,0,160,160,145,100
2,Female,0.0,0.019178,lowest,draw_2,131.073574,17.955243,baseline,0,0,145,100,0,160,160,145,100
3,Female,0.0,0.019178,lowest,draw_3,137.034319,20.486661,baseline,0,0,145,100,0,160,160,145,100
4,Female,0.0,0.019178,lowest,draw_4,141.690229,18.648197,baseline,0,0,145,100,0,160,160,145,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,Male,95.0,125.000000,second,draw_495,97.031167,35.438084,intervention,0,0,110,80,0,130,130,110,80
249996,Male,95.0,125.000000,second,draw_496,106.139058,19.278785,intervention,0,0,110,80,0,130,130,110,80
249997,Male,95.0,125.000000,second,draw_497,105.312300,23.199269,intervention,0,0,110,80,0,130,130,110,80
249998,Male,95.0,125.000000,second,draw_498,99.292330,37.105111,intervention,0,0,110,80,0,130,130,110,80


In [46]:
assert (
    (mean_and_sd_hgb.hgb_upper_mild == mean_and_sd_hgb.hgb_upper_anemic).all() &
    (mean_and_sd_hgb.hgb_lower_severe == mean_and_sd_hgb.hgb_lower_anemic).all()
)
mean_and_sd_hgb = mean_and_sd_hgb.drop(columns=["hgb_upper_anemic", "hgb_lower_anemic"])

In [47]:
assert (
    (mean_and_sd_hgb.hgb_lower_mild == mean_and_sd_hgb.hgb_upper_moderate).all() &
    (mean_and_sd_hgb.hgb_lower_moderate == mean_and_sd_hgb.hgb_upper_severe).all()
)
mean_and_sd_hgb = mean_and_sd_hgb.drop(columns=["hgb_lower_mild", "hgb_lower_moderate"])

In [48]:
def _hemoglobin_distribution_parts_from_mean_sd(mean, sd):
    # NOTE: This is an unusual ensemble distribution. We should add functionality to the
    # EnsembleDistribution class to make this easier.
    x_min = 0
    x_max = 220
    gamma_params = risk_distributions.risk_distributions.Gamma.get_parameters(
        mean=mean, sd=sd
    )
    # NOTE: We have to override these, otherwise Gamma is overly conservative in what values
    # are computable
    # https://github.com/ihmeuw/risk_distributions/issues/61
    gamma_params["x_min"] = x_min
    gamma_params["x_max"] = x_max
    hemoglobin_distribution_gamma_part = risk_distributions.risk_distributions.Gamma(
        gamma_params
    )

    # NOTE: Forced to duplicate https://github.com/ihmeuw/risk_distributions/blob/a9ed9d7e8372590018355012a7a7ffefa87b0819/src/risk_distributions/risk_distributions.py#L428-L434
    # because it doesn't permit the custom x_min and x_max, and these are used in calculating the others
    mgumbel_params = pd.DataFrame({
        "loc": x_max - mean - (np.euler_gamma * np.sqrt(6) / np.pi * sd),
        "scale": np.sqrt(6) / np.pi * sd,
        "x_min": x_min,
        "x_max": x_max,
    })
    hemoglobin_distribution_mgumbel_part = (
        risk_distributions.risk_distributions.MirroredGumbel(mgumbel_params)
    )
    return hemoglobin_distribution_gamma_part, hemoglobin_distribution_mgumbel_part

(
    hemoglobin_distribution_gamma_part,
    hemoglobin_distribution_mgumbel_part,
) = _hemoglobin_distribution_parts_from_mean_sd(mean_and_sd_hgb['mean'], mean_and_sd_hgb.sd)

def cdf(x):
    gamma_cdf = hemoglobin_distribution_gamma_part.cdf(x)
    # NOTE: There is a bug in this CDF function -- it is reversed!
    # https://github.com/ihmeuw/risk_distributions/issues/62
    mgumbel_cdf = 1 - hemoglobin_distribution_mgumbel_part.cdf(x)
    return (
        0.4
        * gamma_cdf
        + 0.6
        * mgumbel_cdf
    )

In [49]:
mean_and_sd_hgb["severe"] = cdf(mean_and_sd_hgb.hgb_upper_severe.copy()) - cdf(mean_and_sd_hgb.hgb_lower_severe.copy())
mean_and_sd_hgb["moderate"] = cdf(mean_and_sd_hgb.hgb_upper_moderate.copy()) - mean_and_sd_hgb["severe"].copy()
mean_and_sd_hgb["mild"] = cdf(mean_and_sd_hgb.hgb_upper_mild.copy()) - mean_and_sd_hgb["moderate"].copy() - mean_and_sd_hgb["severe"].copy()
mean_and_sd_hgb["anemic"] = mean_and_sd_hgb["mild"] + mean_and_sd_hgb["moderate"] + mean_and_sd_hgb["severe"]
mean_and_sd_hgb

,sex,age_start,age_end,wealth_quintile,draw,mean,sd,scenario,pregnant,hgb_lower_severe,hgb_upper_mild,hgb_upper_moderate,hgb_upper_severe,severe,moderate,mild,anemic
0,Female,0.0,0.019178,lowest,draw_0,133.664070,28.489640,baseline,0,0,160,145,100,0.112862,0.522239,0.199909,0.835009
1,Female,0.0,0.019178,lowest,draw_1,133.184863,26.169288,baseline,0,0,160,145,100,0.099261,0.557548,0.208117,0.864926
2,Female,0.0,0.019178,lowest,draw_2,131.073574,17.955243,baseline,0,0,160,145,100,0.048532,0.734690,0.185452,0.968674
3,Female,0.0,0.019178,lowest,draw_3,137.034319,20.486661,baseline,0,0,160,145,100,0.042398,0.586287,0.261777,0.890462
4,Female,0.0,0.019178,lowest,draw_4,141.690229,18.648197,baseline,0,0,160,145,100,0.021629,0.516715,0.313810,0.852154
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,Male,95.0,125.000000,second,draw_495,97.031167,35.438084,intervention,0,0,130,110,80,0.286526,0.341548,0.210403,0.838477
249996,Male,95.0,125.000000,second,draw_496,106.139058,19.278785,intervention,0,0,130,110,80,0.086932,0.463496,0.366191,0.916619
249997,Male,95.0,125.000000,second,draw_497,105.312300,23.199269,intervention,0,0,130,110,80,0.129326,0.423757,0.322755,0.875838
249998,Male,95.0,125.000000,second,draw_498,99.292330,37.105111,intervention,0,0,130,110,80,0.272747,0.323687,0.209075,0.805509


In [50]:
disability_weights = pd.read_hdf('/mnt/team/simulation_science/costeffectiveness/auxiliary_data/GBD_2021/02_processed_data/disability_weight/sequela/all/all.hdf')
disability_weights = disability_weights[disability_weights.healthstate.isin(['anemia_mild', 'anemia_mod', 'anemia_sev'])].set_index('healthstate').filter(like='draw_')
disability_weights.columns.name = 'draw'
disability_weights = disability_weights.stack().rename('disability_weight').reset_index()
disability_weights

,healthstate,draw,disability_weight
0,anemia_mild,draw_0,0.002420
1,anemia_mild,draw_1,0.003172
2,anemia_mild,draw_2,0.002644
3,anemia_mild,draw_3,0.003085
4,anemia_mild,draw_4,0.001845
...,...,...,...
2995,anemia_sev,draw_995,0.174969
2996,anemia_sev,draw_996,0.086798
2997,anemia_sev,draw_997,0.131996
2998,anemia_sev,draw_998,0.124427


In [51]:
mean_and_sd_hgb = mean_and_sd_hgb.merge(
    disability_weights[disability_weights.healthstate == 'anemia_mild'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'mild_dw'}),
    validate="m:1",
).merge(
    disability_weights[disability_weights.healthstate == 'anemia_mod'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'moderate_dw'}),
    validate="m:1",
).merge(
    disability_weights[disability_weights.healthstate == 'anemia_sev'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'severe_dw'}),
    validate="m:1",
)

In [52]:
mean_and_sd_hgb["mild_ylds"] = mean_and_sd_hgb.mild * mean_and_sd_hgb.mild_dw
mean_and_sd_hgb["moderate_ylds"] = mean_and_sd_hgb.moderate * mean_and_sd_hgb.moderate_dw
mean_and_sd_hgb["severe_ylds"] = mean_and_sd_hgb.severe * mean_and_sd_hgb.severe_dw
mean_and_sd_hgb['anemic_ylds'] = mean_and_sd_hgb['mild_ylds'] + mean_and_sd_hgb['moderate_ylds'] + mean_and_sd_hgb['severe_ylds']

In [53]:
index_cols = ['age_start', 'age_end', 'sex', 'draw', "wealth_quintile"]
value_cols = ["mild", "moderate", "severe", "anemic", "mild_ylds", "moderate_ylds", "severe_ylds", "anemic_ylds"]

baseline_anemia = mean_and_sd_hgb[mean_and_sd_hgb.scenario == 'baseline'].set_index(index_cols)[value_cols]
counterfactual_anemia = mean_and_sd_hgb[mean_and_sd_hgb.scenario == 'intervention'].set_index(index_cols)[value_cols]

In [54]:
baseline_anemia.sort_index()

mild  moderate  \
age_start age_end    sex    draw    wealth_quintile                       
0.0       0.019178   Female draw_0  fourth           0.252803  0.456425   
                                    highest          0.274127  0.382060   
                                    lowest           0.199909  0.522239   
                                    middle           0.235473  0.465088   
                                    second           0.218151  0.502540   
...                                                       ...       ...   
95.0      125.000000 Male   draw_99 fourth           0.309215  0.469073   
                                    highest          0.334532  0.460566   
                                    lowest           0.277172  0.484854   
                                    middle           0.302442  0.465940   
                                    second           0.297510  0.475769   

                                                       severe    anemic  \
age_start age_end    sex    draw    wealth_quintile                       
0.0       0.019178   Female draw_0  fourth           0.045826  0.755054   
                                    highest          0.024782  0.680969   
                                    lowest           0.112862  0.835009   
                                    middle           0.060370  0.760930   
                                    second           0.085165  0.805856   
...                                                       ...       ...   
95.0      125.000000 Male   draw_99 fourth           0.141330  0.919618   
                                    highest          0.115949  0.911047   
                                    lowest           0.174130  0.936156   
                                    middle           0.149854  0.918237   
                                    second           0.152991  0.926270   

                                                     mild_ylds  moderate_ylds  \
age_start age_end    sex    draw    wealth_quintile                             
0.0       0.019178   Female draw_0  fourth            0.000612       0.027066   
                                    highest           0.000664       0.022656   
                                    lowest            0.000484       0.030969   
                                    middle            0.000570       0.027580   
                                    second            0.000528       0.029801   
...                                                        ...            ...   
95.0      125.000000 Male   draw_99 fourth            0.001299       0.019551   
                                    highest           0.001405       0.019197   
                                    lowest            0.001164       0.020209   
                                    middle            0.001270       0.019421   
                                    second            0.001250       0.019831   

                                                     severe_ylds  anemic_ylds  
age_start age_end    sex    draw    wealth_quintile                            
0.0       0.019178   Female draw_0  fourth              0.009117     0.036795  
                                    highest             0.004930     0.028250  
                                    lowest              0.022453     0.053906  
                                    middle              0.012010     0.040160  
                                    second              0.016943     0.047272  
...                                                          ...          ...  
95.0      125.000000 Male   draw_99 fourth              0.016788     0.037638  
                                    highest             0.013773     0.034375  
                                    lowest              0.020684     0.042057  
                                    middle              0.017800     0.038492  
                                    second              0.018173     0.039253  

[125000 ro

In [55]:
counterfactual_anemia.sort_index()

mild  moderate  \
age_start age_end    sex    draw    wealth_quintile                       
0.0       0.019178   Female draw_0  fourth           0.253327  0.450738   
                                    highest          0.273763  0.377561   
                                    lowest           0.202885  0.514738   
                                    middle           0.236209  0.459044   
                                    second           0.220001  0.495437   
...                                                       ...       ...   
95.0      125.000000 Male   draw_99 fourth           0.313415  0.464981   
                                    highest          0.337451  0.457096   
                                    lowest           0.286703  0.477961   
                                    middle           0.307318  0.461326   
                                    second           0.304615  0.469525   

                                                       severe    anemic  \
age_start age_end    sex    draw    wealth_quintile                       
0.0       0.019178   Female draw_0  fourth           0.044591  0.748656   
                                    highest          0.024286  0.675610   
                                    lowest           0.107780  0.825403   
                                    middle           0.058538  0.753791   
                                    second           0.081890  0.797329   
...                                                       ...       ...   
95.0      125.000000 Male   draw_99 fourth           0.137514  0.915909   
                                    highest          0.113515  0.908062   
                                    lowest           0.165153  0.929817   
                                    middle           0.145152  0.913797   
                                    second           0.146467  0.920607   

                                                     mild_ylds  moderate_ylds  \
age_start age_end    sex    draw    wealth_quintile                             
0.0       0.019178   Female draw_0  fourth            0.000613       0.026729   
                                    highest           0.000663       0.022390   
                                    lowest            0.000491       0.030524   
                                    middle            0.000572       0.027222   
                                    second            0.000533       0.029380   
...                                                        ...            ...   
95.0      125.000000 Male   draw_99 fourth            0.001316       0.019381   
                                    highest           0.001417       0.019052   
                                    lowest            0.001204       0.019922   
                                    middle            0.001291       0.019229   
                                    second            0.001279       0.019570   

                                                     severe_ylds  anemic_ylds  
age_start age_end    sex    draw    wealth_quintile                            
0.0       0.019178   Female draw_0  fourth              0.008871     0.036213  
                                    highest             0.004831     0.027884  
                                    lowest              0.021442     0.052457  
                                    middle              0.011646     0.039439  
                                    second              0.016291     0.046203  
...                                                          ...          ...  
95.0      125.000000 Male   draw_99 fourth              0.016335     0.037032  
                                    highest             0.013484     0.033954  
                                    lowest              0.019618     0.040744  
                                    middle              0.017242     0.037761  
                                    second              0.017398     0.038248  

[125000 ro

In [56]:
baseline_anemia - counterfactual_anemia

mild  moderate  \
age_start age_end    sex    draw    wealth_quintile                       
0.0       0.019178   Female draw_0  fourth          -0.000525  0.005687   
                                    highest          0.000364  0.004499   
                                    lowest          -0.002976  0.007500   
                                    middle          -0.000736  0.006044   
                                    second          -0.001850  0.007103   
...                                                       ...       ...   
95.0      125.000000 Male   draw_99 fourth          -0.004200  0.004093   
                                    highest         -0.002919  0.003470   
                                    lowest          -0.009531  0.006893   
                                    middle          -0.004876  0.004614   
                                    second          -0.007105  0.006244   

                                                       severe    anemic  \
age_start age_end    sex    draw    wealth_quintile                       
0.0       0.019178   Female draw_0  fourth           0.001235  0.006398   
                                    highest          0.000496  0.005359   
                                    lowest           0.005082  0.009606   
                                    middle           0.001832  0.007139   
                                    second           0.003274  0.008527   
...                                                       ...       ...   
95.0      125.000000 Male   draw_99 fourth           0.003816  0.003709   
                                    highest          0.002434  0.002985   
                                    lowest           0.008977  0.006339   
                                    middle           0.004702  0.004441   
                                    second           0.006524  0.005663   

                                                        mild_ylds  \
age_start age_end    sex    draw    wealth_quintile                 
0.0       0.019178   Female draw_0  fourth          -1.270008e-06   
                                    highest          8.819852e-07   
                                    lowest          -7.203445e-06   
                                    middle          -1.781471e-06   
                                    second          -4.478496e-06   
...                                                           ...   
95.0      125.000000 Male   draw_99 fourth          -1.764184e-05   
                                    highest         -1.226141e-05   
                                    lowest          -4.003201e-05   
                                    middle          -2.048020e-05   
                                    second          -2.984204e-05   

                                                     moderate_ylds  \
age_start age_end    sex    draw    wealth_quintile                  
0.0       0.019178   Female draw_0  fourth                0.000337   
                                    highest               0.000267   
                                    lowest                0.000445   
                                    middle                0.000358   
                                    second                0.000421   
...                                                            ...   
95.0      125.000000 Male   draw_99 fourth                0.000171   
                                    highest               0.000145   
                                    lowest                0.000287   
                                    middle                0.000192   
                                    second                0.000260   

                                                     severe_ylds  anemic_ylds  
age_start age_end    sex    draw    wealth_quintile                            
0.0       0.019178   Female draw_0  fourth              0.000246     0.000582  
                                    highest             

In [57]:
# BUT our counterfactual is only in a world where everyone is iron-responsive.
# Cleaned this up from https://github.com/ihmeuw/vivarium_research_lsff/blob/1cb465a752d299401ae366db537dc8d557162184/multiplication_models/iron_model_U5.ipynb,
# but have not checked it in extreme detail.
iron_responsive_anemia_sequelae = [
    gbd_mapping.sequelae.mild_anemia_due_to_schistosomiasis,
    gbd_mapping.sequelae.moderate_anemia_due_to_schistosomiasis,
    gbd_mapping.sequelae.severe_anemia_due_to_schistosomiasis,
    gbd_mapping.sequelae.mild_anemia_due_to_hookworm_disease,
    gbd_mapping.sequelae.moderate_anemia_due_to_hookworm_disease,
    gbd_mapping.sequelae.severe_anemia_due_to_hookworm_disease,
    gbd_mapping.sequelae.mild_anemia_due_to_other_neglected_tropical_diseases,
    gbd_mapping.sequelae.moderate_anemia_due_to_other_neglected_tropical_diseases,
    gbd_mapping.sequelae.severe_anemia_due_to_other_neglected_tropical_diseases,
    gbd_mapping.sequelae.mild_anemia_due_to_maternal_hemorrhage,
    gbd_mapping.sequelae.moderate_anemia_due_to_maternal_hemorrhage,
    gbd_mapping.sequelae.severe_anemia_due_to_maternal_hemorrhage,
    gbd_mapping.sequelae.mild_iron_deficiency_anemia,
    gbd_mapping.sequelae.moderate_iron_deficiency_anemia,
    gbd_mapping.sequelae.severe_iron_deficiency_anemia,
    gbd_mapping.sequelae.mild_anemia_due_to_other_infectious_diseases,
    gbd_mapping.sequelae.moderate_anemia_due_to_other_infectious_diseases,
    gbd_mapping.sequelae.severe_anemia_due_to_other_infectious_diseases,
    gbd_mapping.sequelae.menstrual_disorders_with_mild_anemia,
    gbd_mapping.sequelae.menstrual_disorders_with_moderate_anemia,
    gbd_mapping.sequelae.menstrual_disorders_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_hypertension_with_mild_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_hypertension_with_moderate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_hypertension_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_glomerulonephritis_with_mild_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_glomerulonephritis_with_moderate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_glomerulonephritis_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_other_and_unspecified_causes_with_mild_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_other_and_unspecified_causes_with_moderate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_other_and_unspecified_causes_with_severe_anemia,
    gbd_mapping.sequelae.uterine_fibroids_symptomatic_with_mild_anemia,
    gbd_mapping.sequelae.uterine_fibroids_symptomatic_with_moderate_anemia,
    gbd_mapping.sequelae.uterine_fibroids_symptomatic_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_hypertension_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_hypertension_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_hypertension_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_hypertension_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_hypertension_with_moderate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_hypertension_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_moderate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_moderate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_severe_anemia,
    gbd_mapping.sequelae.mildly_symptomatic_pud_with_mild_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_pud_with_mild_anemia,
    gbd_mapping.sequelae.mildly_symptomatic_pud_with_moderate_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_pud_with_moderate_anemia,
    gbd_mapping.sequelae.mildly_symptomatic_pud_with_severe_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_pud_with_severe_anemia,
    gbd_mapping.sequelae.asymptomatic_pud_with_mild_anemia,
    gbd_mapping.sequelae.asymptomatic_pud_with_moderate_anemia,
    gbd_mapping.sequelae.asymptomatic_pud_with_severe_anemia,
    gbd_mapping.sequelae.mildy_symptomatic_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.mildly_symptomatic_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.mildy_symptomatic_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.asymptomatic_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.asymptomatic_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.asymptomatic_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_1_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_2_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_1_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_2_diabetes_mellitus_with_mdoerate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_1_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_2_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_mdoerate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.vitamin_a_deficiency_with_mild_anemia,
    gbd_mapping.sequelae.vitamin_a_deficiency_with_moderate_anemia,
    gbd_mapping.sequelae.vitamin_a_deficiency_with_severe_anemia,
    gbd_mapping.sequelae.ulcerative_colitis_with_mild_anemia,
    gbd_mapping.sequelae.ulcerative_colitis_with_moderate_anemia,
    gbd_mapping.sequelae.ulcerative_colitis_with_severe_anemia,
    gbd_mapping.sequelae.crohns_disease_with_mild_anemia,
    gbd_mapping.sequelae.crohns_disease_with_moderate_anemia,
    gbd_mapping.sequelae.crohns_disease_with_severe_anemia,
    gbd_mapping.sequelae.complicated_pud_with_mild_anemia,
    gbd_mapping.sequelae.complicated_pud_with_moderate_anemia,
    gbd_mapping.sequelae.complicated_pud_with_severe_anemia,
    gbd_mapping.sequelae.complicated_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.complicated_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.complicated_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_pud_with_mild_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_pud_with_moderate_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_pud_with_severe_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_1_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_2_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_hypertension_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_glomerulonephritis_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_other_and_unspecified_causes_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_1_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_2_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_hypertension_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_glomerulonephritis_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_other_and_unspecified_causes_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_1_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_2_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_hypertension_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_glomerulonephritis_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_other_and_unspecified_causes_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_b_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_b_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_b_decompensated_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_c_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_c_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_c_decompensated_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_alcohol_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_alcohol_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_alcohol_decompensated_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_other_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_other_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_other_decompensated_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_nash_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_nash_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_nash_decompensated_with_severe_anemia,
]

In [58]:
non_iron_responsive_anemia_sequelae = [
    gbd_mapping.sequelae.mild_anemia_due_to_other_hemoglobinopathies_and_hemolytic_anemias,
    gbd_mapping.sequelae.moderate_anemia_due_to_other_hemoglobinopathies_and_hemolytic_anemias,
    gbd_mapping.sequelae.severe_anemia_due_to_other_hemoglobinopathies_and_hemolytic_anemias,
    gbd_mapping.sequelae.mild_anemia_due_to_b_thalassemia_trait,
    gbd_mapping.sequelae.moderate_anemia_due_to_b_thalassemia_trait,
    gbd_mapping.sequelae.severe_anemia_due_to_b_thalassemia_trait,
    gbd_mapping.sequelae.mild_anemia_due_to_hemoglobin_e_trait,
    gbd_mapping.sequelae.moderate_anemia_due_to_hemoglobin_e_trait,
    gbd_mapping.sequelae.severe_anemia_due_to_hemoglobin_e_trait,
    gbd_mapping.sequelae.mild_anemia_due_to_sickle_cell_trait,
    gbd_mapping.sequelae.moderate_anemia_due_to_sickle_cell_trait,
    gbd_mapping.sequelae.severe_anemia_due_to_sickle_cell_trait,
    gbd_mapping.sequelae.hemizygous_g6pd_deficiency_with_mild_anemia,
    gbd_mapping.sequelae.hemizygous_g6pd_deficiency_with_moderate_anemia,
    gbd_mapping.sequelae.hemizygous_g6pd_deficiency_with_severe_anemia,
    gbd_mapping.sequelae.mild_anemia_due_to_malaria_parasitemia_pfpr,
    gbd_mapping.sequelae.moderate_anemia_due_to_malaria_parasitemia_pfpr,
    gbd_mapping.sequelae.severe_anemia_due_to_malaria_parasitemia_pfpr,
    gbd_mapping.sequelae.mild_anemia_due_to_homozygous_sickle_cell_and_severe_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.moderate_anemia_due_to_homozygous_sickle_cell_and_severe_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.severe_anemia_due_to_homozygous_sickle_cell_and_severe_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.mild_anemia_due_to_hemoglobin_sc_disease,
    gbd_mapping.sequelae.moderate_anemia_due_to_hemoglobin_sc_disease,
    gbd_mapping.sequelae.severe_anemia_due_to_hemoglobin_sc_disease,
    gbd_mapping.sequelae.mild_anemia_due_to_mild_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.moderate_anemia_due_to_mild_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.severe_anemia_due_to_mild_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.severe_malaria_with_mild_anemia,
    gbd_mapping.sequelae.severe_malaria_with_moderate_anemia,
    gbd_mapping.sequelae.severe_malaria_with_severe_anemia,
    gbd_mapping.sequelae.mild_malaria_with_mild_anemia,
    gbd_mapping.sequelae.mild_malaria_with_moderate_anemia,
    gbd_mapping.sequelae.mild_malaria_with_severe_anemia,
    gbd_mapping.sequelae.moderate_malaria_with_mild_anemia,
    gbd_mapping.sequelae.moderate_malaria_with_moderate_anemia,
    gbd_mapping.sequelae.moderate_malaria_with_severe_anemia,
    gbd_mapping.sequelae.early_hiv_with_mild_anemia,
    gbd_mapping.sequelae.early_hiv_with_moderate_anemia,
    gbd_mapping.sequelae.early_hiv_with_severe_anemia,
    gbd_mapping.sequelae.symptomatic_hiv_with_mild_anemia,
    gbd_mapping.sequelae.symptomatic_hiv_with_moderate_anemia,
    gbd_mapping.sequelae.symptomatic_hiv_with_severe_anemia,
    gbd_mapping.sequelae.hiv_aids_with_antiretroviral_treatment_with_mild_anemia,
    gbd_mapping.sequelae.hiv_aids_with_antiretroviral_treatment_with_moderate_anemia,
    gbd_mapping.sequelae.hiv_aids_with_antiretroviral_treatment_with_severe_anemia,
    gbd_mapping.sequelae.aids_with_mild_anemia,
    gbd_mapping.sequelae.aids_with_moderate_anemia,
    gbd_mapping.sequelae.aids_with_severe_anemia,
    gbd_mapping.sequelae.hiv_aids_drug_susceptible_tuberculosis_with_mild_anemia,
    gbd_mapping.sequelae.hiv_aids_drug_susceptible_tuberculosis_with_moderate_anemia,
    gbd_mapping.sequelae.hiv_aids_drug_susceptible_tuberculosis_with_severe_anemia,
    gbd_mapping.sequelae.hiv_aids_multidrug_resistant_tuberculosis_without_extensive_drug_resistance_with_mild_anemia,
    gbd_mapping.sequelae.hiv_aids_multidrug_resistant_tuberculosis_without_extensive_drug_resistance_with_moderate_anemia,
    gbd_mapping.sequelae.hiv_aids_multidrug_resistant_tuberculosis_without_extensive_drug_resistance_with_severe_anemia,
    gbd_mapping.sequelae.hiv_aids_extensively_drug_resistant_tuberculosis_with_mild_anemia,
    gbd_mapping.sequelae.hiv_aids_extensively_drug_resistant_tuberculosis_with_moderate_anemia,
    gbd_mapping.sequelae.hiv_aids_extensively_drug_resistant_tuberculosis_with_severe_anemia,
    gbd_mapping.sequelae.mild_anemia_due_to_malaria_vivax_pvpr,
    gbd_mapping.sequelae.moderate_anemia_due_to_malaria_vivax_pvpr,
    gbd_mapping.sequelae.severe_anemia_due_to_malaria_vivax_pvpr,
]

In [59]:
len(iron_responsive_anemia_sequelae)

138

In [60]:
len(non_iron_responsive_anemia_sequelae)

60

In [61]:
loguru.logger.disable("vivarium_inputs.validation.raw")

In [62]:
iron_responsive_prevalence = None

for sequela in tqdm(iron_responsive_anemia_sequelae):
    try:
        sequela_prevalence = vivarium_inputs.get_measure(sequela, "prevalence", location.title()).droplevel(["location", "year_start", "year_end"])
    except DataDoesNotExistError as e:
        assert 'zero' in str(e)
        continue
    except DataAbnormalError as e:
        assert 'zero' in str(e)
        continue

    if iron_responsive_prevalence is None:
        iron_responsive_prevalence = sequela_prevalence
    else:
        iron_responsive_prevalence += sequela_prevalence

  0%|          | 0/138 [00:00<?, ?it/s]

In [63]:
non_iron_responsive_prevalence = None

for sequela in tqdm(non_iron_responsive_anemia_sequelae):
    try:
        sequela_prevalence = vivarium_inputs.get_measure(sequela, "prevalence", location.title()).droplevel(["location", "year_start", "year_end"])
    except DataDoesNotExistError as e:
        assert 'zero' in str(e)
        continue
    except DataAbnormalError as e:
        assert 'zero' in str(e)
        continue

    if non_iron_responsive_prevalence is None:
        non_iron_responsive_prevalence = sequela_prevalence
    else:
        non_iron_responsive_prevalence += sequela_prevalence

  0%|          | 0/60 [00:00<?, ?it/s]

In [64]:
loguru.logger.enable("vivarium_inputs.validation.raw")

In [65]:
iron_responsive_prevalence.columns.name = "draw"
iron_responsive_prevalence = iron_responsive_prevalence.stack().pipe(lambda s: s[s.index.get_level_values("draw").isin(DRAWS)])
iron_responsive_prevalence

sex     age_start  age_end     draw    
Female  0.0        0.019178    draw_0      0.811025
                               draw_1      0.698829
                               draw_2      0.727310
                               draw_3      0.843570
                               draw_4      0.832664
                                             ...   
Male    95.0       125.000000  draw_495    0.655482
                               draw_496    0.787855
                               draw_497    0.838763
                               draw_498    0.804139
                               draw_499    0.781115
Length: 25000, dtype: float64

In [66]:
non_iron_responsive_prevalence.columns.name = "draw"
non_iron_responsive_prevalence = non_iron_responsive_prevalence.stack().pipe(lambda s: s[s.index.get_level_values("draw").isin(DRAWS)])
non_iron_responsive_prevalence

sex     age_start  age_end     draw    
Female  0.0        0.019178    draw_0      0.118866
                               draw_1      0.107433
                               draw_2      0.138745
                               draw_3      0.085699
                               draw_4      0.138585
                                             ...   
Male    95.0       125.000000  draw_495    0.112535
                               draw_496    0.142128
                               draw_497    0.206873
                               draw_498    0.146891
                               draw_499    0.133303
Length: 25000, dtype: float64

In [67]:
iron_responsive_proportion = iron_responsive_prevalence / (iron_responsive_prevalence + non_iron_responsive_prevalence)
iron_responsive_proportion

sex     age_start  age_end     draw    
Female  0.0        0.019178    draw_0      0.872172
                               draw_1      0.866752
                               draw_2      0.839797
                               draw_3      0.907778
                               draw_4      0.857313
                                             ...   
Male    95.0       125.000000  draw_495    0.853473
                               draw_496    0.847171
                               draw_497    0.802156
                               draw_498    0.845546
                               draw_499    0.854221
Length: 25000, dtype: float64

In [68]:
assert (iron_responsive_proportion <= 1).all()

In [69]:
iron_responsive_proportion.sort_values()

sex     age_start  age_end    draw    
Male    10.0       15.000000  draw_352    0.388719
                              draw_123    0.405524
                              draw_74     0.405664
                              draw_25     0.411639
                              draw_343    0.427580
                                            ...   
        0.0        0.019178   draw_22     0.924246
                              draw_335    0.924433
                              draw_160    0.924635
Female  0.0        0.019178   draw_459    0.927894
                              draw_256    0.931564
Length: 25000, dtype: float64

In [70]:
# NOTE: I am pretty sure this is correct, but it is quite difficult to think through *why*.

# First, observe that people in the population who start as non-anemic never factor into
# any of these metrics. If they started non-anemic, our counterfactual can only shift them up,
# so they did not change anemia categories between scenarios and hence have no importance to
# anemia prevalence or YLDs.

# So you can think of our hemoglobin distributions as only being of interest in the part
# of them below the anemia threshold.

# *Within* this subpopulation, we make the assumption that iron-responsive and non-iron-responsive
# anemic people have the same distributions of hemoglobin. This is probably not true, but GBD doesn't
# give us anything better.

# So you can think of our original distribution as a mixture of two parts, which are the same.
# Then we shift one of those parts (the iron-responsive part) and calculate all these stats from
# that new distribution.

# Our *actual* result should be about a mixture distribution that has the non-iron-responsive part
# the same as in baseline, with the shifted iron-responsive part.
# For all these metrics, it is pretty straightforward to see that the metric in such a mixture is
# just a weighted average of the metrics in each part, since they all depend on CDFs which combine
# this way.

counterfactual_anemia_accounting_for_non_response = (
    counterfactual_anemia.mul(iron_responsive_proportion, axis=0) +
    baseline_anemia.mul(1 - iron_responsive_proportion, axis=0)
)
counterfactual_anemia_accounting_for_non_response

mild  moderate  \
age_start age_end    sex    draw    wealth_quintile                       
0.0       0.019178   Female draw_0  fourth           0.253260  0.451465   
                                    highest          0.273810  0.378136   
                                    lowest           0.202504  0.515697   
                                    middle           0.236115  0.459817   
                                    second           0.219765  0.496345   
...                                                       ...       ...   
95.0      125.000000 Male   draw_99 fourth           0.312665  0.465711   
                                    highest          0.336930  0.457715   
                                    lowest           0.285002  0.479192   
                                    middle           0.306448  0.462150   
                                    second           0.303347  0.470640   

                                                       severe    anemic  \
age_start age_end    sex    draw    wealth_quintile                       
0.0       0.019178   Female draw_0  fourth           0.044749  0.749474   
                                    highest          0.024349  0.676295   
                                    lowest           0.108429  0.826631   
                                    middle           0.058772  0.754703   
                                    second           0.082309  0.798419   
...                                                       ...       ...   
95.0      125.000000 Male   draw_99 fourth           0.138195  0.916572   
                                    highest          0.113950  0.908595   
                                    lowest           0.166756  0.930949   
                                    middle           0.145992  0.914589   
                                    second           0.147631  0.921618   

                                                     mild_ylds  moderate_ylds  \
age_start age_end    sex    draw    wealth_quintile                             
0.0       0.019178   Female draw_0  fourth            0.000613       0.026772   
                                    highest           0.000663       0.022424   
                                    lowest            0.000490       0.030581   
                                    middle            0.000572       0.027267   
                                    second            0.000532       0.029434   
...                                                        ...            ...   
95.0      125.000000 Male   draw_99 fourth            0.001313       0.019411   
                                    highest           0.001415       0.019078   
                                    lowest            0.001197       0.019973   
                                    middle            0.001287       0.019263   
                                    second            0.001274       0.019617   

                                                     severe_ylds  anemic_ylds  
age_start age_end    sex    draw    wealth_quintile                            
0.0       0.019178   Female draw_0  fourth              0.008902     0.036287  
                                    highest             0.004844     0.027930  
                                    lowest              0.021571     0.052642  
                                    middle              0.011692     0.039531  
                                    second              0.016375     0.046340  
...                                                          ...          ...  
95.0      125.000000 Male   draw_99 fourth              0.016416     0.037140  
                                    highest             0.013536     0.034029  
                                    lowest              0.019808     0.040978  
                                    middle              0.017342     0.037892  
                                    second              0.017536     0.038427  

[125000 ro

In [71]:
assert ((baseline_anemia - counterfactual_anemia_accounting_for_non_response).anemic_ylds > 0).all()

In [72]:
baseline_ylds = (baseline_anemia.anemic_ylds.unstack("draw").mean(axis=1) * non_pregnant_pop)
baseline_ylds

age_start  age_end     sex     wealth_quintile
0.0        0.019178    Female  fourth             424.729902
                               highest            291.089988
                               lowest             736.111714
                               middle             521.550200
                               second             650.757670
                                                     ...    
95.0       125.000000  Male    fourth              76.888359
                               highest             74.821911
                               lowest              89.386189
                               middle              73.428445
                               second              72.903842
Length: 250, dtype: float64

In [73]:
intervention_ylds = (counterfactual_anemia_accounting_for_non_response.anemic_ylds.unstack("draw").mean(axis=1) * non_pregnant_pop)

In [74]:
baseline_ylds.groupby(["wealth_quintile"]).sum() - intervention_ylds.groupby(["wealth_quintile"]).sum()

wealth_quintile
fourth     14112.720064
highest     8264.440896
lowest     36104.236694
middle     17569.786970
second     25944.580011
dtype: float64

In [75]:
ylds = pd.concat([
    baseline_ylds.rename("value").reset_index().assign(scenario="baseline"),
    intervention_ylds.rename("value").reset_index().assign(scenario="intervention")
], ignore_index=True)
ylds

,age_start,age_end,sex,wealth_quintile,value,scenario
0,0.0,0.019178,Female,fourth,424.729902,baseline
1,0.0,0.019178,Female,highest,291.089988,baseline
2,0.0,0.019178,Female,lowest,736.111714,baseline
3,0.0,0.019178,Female,middle,521.550200,baseline
4,0.0,0.019178,Female,second,650.757670,baseline
...,...,...,...,...,...,...
495,95.0,125.000000,Male,fourth,75.850598,intervention
496,95.0,125.000000,Male,highest,74.056934,intervention
497,95.0,125.000000,Male,lowest,87.028308,intervention
498,95.0,125.000000,Male,middle,72.260735,intervention


In [76]:
results_dir = f'./results/{vehicle.lower()}/{location.lower()}/{intervention_scenario.lower()}'

In [77]:
path = f'{results_dir}/ylds.parquet'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ylds.to_parquet(path)

In [78]:
baseline_anemia_prevalence = baseline_anemia['anemic'].unstack("draw").mean(axis=1)
baseline_anemia_prevalence

age_start  age_end     sex     wealth_quintile
0.0        0.019178    Female  fourth             0.828338
                               highest            0.748522
                               lowest             0.900708
                               middle             0.835819
                               second             0.876689
                                                    ...   
95.0       125.000000  Male    fourth             0.891835
                               highest            0.880907
                               lowest             0.910303
                               middle             0.890964
                               second             0.899321
Length: 250, dtype: float64

In [79]:
baseline_anemia_cases = baseline_anemia_prevalence.mul(non_pregnant_pop, axis=0)
baseline_anemia_cases

age_start  age_end     sex     wealth_quintile
0.0        0.019178    Female  fourth             11611.609394
                               highest             9753.105130
                               lowest             15276.458404
                               middle             13328.773023
                               second             14808.048479
                                                      ...     
95.0       125.000000  Male    fourth              1586.629929
                               highest             1671.057535
                               lowest              1681.299915
                               middle              1480.290163
                               second              1453.600055
Length: 250, dtype: float64

In [80]:
baseline_anemia_cases.sum() / non_pregnant_pop.sum()

0.45867407015636436

In [81]:
baseline_anemia_cases.groupby(["wealth_quintile"]).sum() / non_pregnant_pop.groupby(["wealth_quintile"]).sum()

wealth_quintile
fourth     0.434832
highest    0.350655
lowest     0.556781
middle     0.454171
second     0.498840
dtype: float64

In [82]:
intervention_anemia_prevalence = counterfactual_anemia_accounting_for_non_response['anemic'].unstack("draw").mean(axis=1)
intervention_anemia_prevalence

age_start  age_end     sex     wealth_quintile
0.0        0.019178    Female  fourth             0.822986
                               highest            0.743238
                               lowest             0.894374
                               middle             0.829923
                               second             0.870501
                                                    ...   
95.0       125.000000  Male    fourth             0.888494
                               highest            0.878136
                               lowest             0.904701
                               middle             0.887005
                               second             0.894267
Length: 250, dtype: float64

In [83]:
anemia_prevalence = pd.concat([
    baseline_anemia_prevalence.rename("value").reset_index().assign(scenario="baseline"),
    intervention_anemia_prevalence.rename("value").reset_index().assign(scenario="intervention")
], ignore_index=True)
anemia_prevalence

,age_start,age_end,sex,wealth_quintile,value,scenario
0,0.0,0.019178,Female,fourth,0.828338,baseline
1,0.0,0.019178,Female,highest,0.748522,baseline
2,0.0,0.019178,Female,lowest,0.900708,baseline
3,0.0,0.019178,Female,middle,0.835819,baseline
4,0.0,0.019178,Female,second,0.876689,baseline
...,...,...,...,...,...,...
495,95.0,125.000000,Male,fourth,0.888494,intervention
496,95.0,125.000000,Male,highest,0.878136,intervention
497,95.0,125.000000,Male,lowest,0.904701,intervention
498,95.0,125.000000,Male,middle,0.887005,intervention


In [84]:
path = f'{results_dir}/anemia_prevalence.parquet'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
anemia_prevalence.to_parquet(path)

In [85]:
intervention_anemia_cases = intervention_anemia_prevalence.mul(non_pregnant_pop, axis=0)
intervention_anemia_cases

age_start  age_end     sex     wealth_quintile
0.0        0.019178    Female  fourth             11536.584576
                               highest             9684.248311
                               lowest             15169.022189
                               middle             13234.743021
                               second             14703.538673
                                                      ...     
95.0       125.000000  Male    fourth              1580.686026
                               highest             1665.801103
                               lowest              1670.953722
                               middle              1473.711399
                               second              1445.429859
Length: 250, dtype: float64

In [86]:
intervention_anemia_cases.sum() / non_pregnant_pop.sum()

0.44896102525225096

In [87]:
intervention_anemia_cases.groupby(["wealth_quintile"]).sum() / non_pregnant_pop.groupby(["wealth_quintile"]).sum()

wealth_quintile
fourth     0.427379
highest    0.345507
lowest     0.541633
middle     0.445316
second     0.486771
dtype: float64

In [88]:
(baseline_anemia_cases.groupby(["wealth_quintile"]).sum() - intervention_anemia_cases.groupby(["wealth_quintile"]).sum()).map(lambda x: f'{round(x):,.0f}')

wealth_quintile
fourth     336,497
highest    231,349
lowest     670,752
middle     396,040
second     537,272
dtype: object

In [89]:
anemia_cases = pd.concat([
    baseline_anemia_cases.rename("value").reset_index().assign(scenario="baseline"),
    intervention_anemia_cases.rename("value").reset_index().assign(scenario="intervention")
], ignore_index=True)
anemia_cases

,age_start,age_end,sex,wealth_quintile,value,scenario
0,0.0,0.019178,Female,fourth,11611.609394,baseline
1,0.0,0.019178,Female,highest,9753.105130,baseline
2,0.0,0.019178,Female,lowest,15276.458404,baseline
3,0.0,0.019178,Female,middle,13328.773023,baseline
4,0.0,0.019178,Female,second,14808.048479,baseline
...,...,...,...,...,...,...
495,95.0,125.000000,Male,fourth,1580.686026,intervention
496,95.0,125.000000,Male,highest,1665.801103,intervention
497,95.0,125.000000,Male,lowest,1670.953722,intervention
498,95.0,125.000000,Male,middle,1473.711399,intervention


In [90]:
path = f'{results_dir}/anemia_cases.parquet'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
anemia_cases.to_parquet(path)